## Setup

In [ ]:
#TODO - clean up code, fix imports, hyperparam tuning, add more features, experiment with custom kernel
#TODO!! - will need to convert SVM implementation to cuML for GPU, convert to CuPy arrays

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report
nlp = spacy.load("en_core_web_sm")

##Load data

In [ ]:
data_path = "/content/drive/MyDrive/U of Manchester/Relation Extraction Project/data" #change

def load_data(filename):
    with open(os.path.join(data_path, filename), 'r') as f:
        data = json.load(f)
    return data

train_data = load_data("retacred_train.json")
dev_data = load_data("retacred_dev.json")
test_data = load_data("retacred_test.json")

#uncomment to test on original tacred data
# train_data = load_data("tacred_train.json")
# dev_data = load_data("tacred_dev.json")
# test_data = load_data("tacred_test.json")

print(f"Train: {len(train_data)}, Dev: {len(dev_data)}, Test: {len(test_data)}")

Train: 58465, Dev: 19584, Test: 13418


## Functions to extract features

In [ ]:
def get_words_and_pos(tokens, pos_tags, subj_start, subj_end, obj_start, obj_end):
    words = []
    pos = []
    #if subject is before object
    if subj_end < obj_start:

        #mark e1 for subject
        for i in range(subj_start, subj_end + 1):
            position = i - subj_start + 1
            words.append(f"{tokens[i]}(e1-{position})")
            pos.append(f"{pos_tags[i]}(e1-{position})")

        #all words between
        words.extend(tokens[subj_end + 1: obj_start])
        pos.extend(pos_tags[subj_end + 1: obj_start])

        # mark e2 for object
        for i in range(obj_start, obj_end + 1):
            position = i - obj_start + 1
            words.append(f"{tokens[i]}(e2-{position})")
            pos.append(f"{pos_tags[i]}(e2-{position})")


    # if object comes before subject
    elif obj_end < subj_start:
        # mark e2 for object
        for i in range(obj_start, obj_end + 1):
            position = i - obj_start + 1
            words.append(f"{tokens[i]}(e2-{position})")
            pos.append(f"{pos_tags[i]}(e2-{position})")

        #words in between
        words.extend(tokens[obj_end + 1: subj_start])
        pos.extend(pos_tags[obj_end + 1: subj_start])

        # mark e1 for subject
        for i in range(subj_start, subj_end + 1):
            position = i - subj_start + 1
            words.append(f"{tokens[i]}(e1-{position})")
            pos.append(f"{pos_tags[i]}(e1-{position})")

    return words, pos

In [ ]:
def get_entity_types(tokens, subj_start, subj_end, obj_start, obj_end, subj_type, obj_type):
    #full mention for the subject
    subj_tokens = tokens[subj_start:subj_end + 1]
    subj_mention = " ".join(subj_tokens)

    #full mention for the object
    obj_tokens = tokens[obj_start:obj_end + 1]
    obj_mention = " ".join(obj_tokens)

    #2 possible orders of mentions
    mentions = []
    if subj_start < obj_start:
        mentions.append((subj_mention, subj_type))
        mentions.append((obj_mention, obj_type))
    else:
        mentions.append((obj_mention, obj_type))
        mentions.append((subj_mention, subj_type))

    return mentions


In [ ]:
def get_entity_mention_type_spacy(tokens, subj_start, subj_end, obj_start, obj_end):
    sentence = " ".join(tokens)
    doc = nlp(sentence)

    # full mention and text span for subject and object
    subj_mention = " ".join(tokens[subj_start:subj_end + 1])
    obj_mention = " ".join(tokens[obj_start:obj_end + 1])

    def get_mention_type(mention):
        for ent in doc.ents:
            if mention in ent.text:
                # Use spaCy's NER labels to determine NAME or NOMINAL
                if ent.label_ in {"PERSON", "ORG", "GPE", "LOC", "PRODUCT", "EVENT", "WORK_OF_ART", "LAW", "LANGUAGE"}:
                    return "NAME"
        # If no NER label, check if it is a pronoun
        if mention.lower() in {"i", "me", "you", "he", "him", "she", "her", "it",
                               "we", "us", "they", "them", "my", "your", "his",
                               "its", "our", "their", "mine", "yours", "hers",
                               "ours", "theirs"}:
            return "PRONOUN"
        return "NOMINAL"

    # get mention types
    subj_mention_type = get_mention_type(subj_mention)
    obj_mention_type = get_mention_type(obj_mention)

    # find order of mentions for this sentence
    mention_types = []
    if subj_start < obj_start:
        mention_types.append((subj_mention, subj_mention_type))
        mention_types.append((obj_mention, obj_mention_type))
    else:
        mention_types.append((obj_mention, obj_mention_type))
        mention_types.append((subj_mention, subj_mention_type))

    return mention_types


##Extract features from data

In [ ]:
def extract_features(data):
    tokens = data['token']
    pos_tags = data['stanford_pos']
    subj_start = data['subj_start']
    subj_end = data['subj_end']
    obj_start = data['obj_start']
    obj_end = data['obj_end']
    subj_type = data['subj_type']
    obj_type = data['obj_type']

    features = {}

    #label
    features['label'] = data['relation']

    #feature #1: words
    #feature #2: pos
    words, pos = get_words_and_pos(tokens, pos_tags, subj_start, subj_end, obj_start, obj_end)
    features['words'] = ' '.join(words)
    features['pos'] = ' '.join(pos)

    #feature #3: entity type
    entity_types = get_entity_types(tokens, subj_start, subj_end, obj_start, obj_end, subj_type, obj_type)
    features['entity_types'] = entity_types

    #feature #4: entity mention type
    entity_mention_type = get_entity_mention_type_spacy(tokens, subj_start, subj_end, obj_start, obj_end)
    features['entity_mention_type'] = entity_mention_type


    #DEBGUGGINS
    # print("WORDS: ", words)
    # print("POS: ", pos)
    # print("ENTITY TYPES: ", entity_types)
    # print("ENTITY MENTION TYPE: ", entity_mention_type)

    return features

features = extract_features(train_data[0])

In [ ]:
features

{'label': 'org:founded_by',
 'words': 'Tom(e2-1) Thabane(e2-2) resigned in October last year to form the All(e1-1) Basotho(e1-2) Convention(e1-3)',
 'pos': 'NNP(e2-1) NNP(e2-2) VBD IN NNP JJ NN TO VB DT DT(e1-1) NNP(e1-2) NNP(e1-3)',
 'entity_types': [('Tom Thabane', 'PERSON'),
  ('All Basotho Convention', 'ORGANIZATION')],
 'entity_mention_type': [('Tom Thabane', 'NAME'),
  ('All Basotho Convention', 'NAME')]}

In [ ]:
from tqdm import tqdm
import pickle

# train_features_file = 'train_features.pkl'
# dev_features_file = 'dev_features.pkl'
# test_features_file = 'test_features.pkl'

train_features_file = '/content/drive/MyDrive/U of Manchester/Relation Extraction Project/train_features.pkl'
dev_features_file = '/content/drive/MyDrive/U of Manchester/Relation Extraction Project/dev_features.pkl'
test_features_file = '/content/drive/MyDrive/U of Manchester/Relation Extraction Project/test_features.pkl'

#check if we already have the features saved...
if os.path.exists(train_features_file) and os.path.exists(dev_features_file) and os.path.exists(test_features_file):
    print("loading saved features...")
    with open(train_features_file, 'rb') as f:
        train_features = pickle.load(f)
    with open(dev_features_file, 'rb') as f:
        dev_features = pickle.load(f)
    with open(test_features_file, 'rb') as f:
        test_features = pickle.load(f)

#if not extract features
else:
    train_features = [extract_features(item) for item in tqdm(train_data, desc="Processing training data")]
    dev_features = [extract_features(item) for item in tqdm(dev_data, desc="Processing dev data")]
    test_features = [extract_features(item) for item in tqdm(test_data, desc="Processing test data")]

    with open(train_features_file, 'wb') as f:
        pickle.dump(train_features, f)
    with open(dev_features_file, 'wb') as f:
        pickle.dump(dev_features, f)
    with open(test_features_file, 'wb') as f:
        pickle.dump(test_features, f)
    print("saved processed features")



loading saved features...


##Vectorization (currently TF-IDF) NOTE: paper did binary features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer_words = TfidfVectorizer()
train_vectors_words = vectorizer_words.fit_transform([item['words'] for item in train_features])
dev_vectors_words = vectorizer_words.transform([item['words'] for item in dev_features])
test_vectors_words = vectorizer_words.transform([item['words'] for item in test_features])


In [ ]:
vectorizer_pos = TfidfVectorizer()
train_vectors_pos = vectorizer_pos.fit_transform([item['pos'] for item in train_features])
dev_vectors_pos = vectorizer_pos.transform([item['pos'] for item in dev_features])
test_vectors_pos = vectorizer_pos.transform([item['pos'] for item in test_features])


In [ ]:
vectorizer_entity_types = TfidfVectorizer()
train_vectors_entity_types = vectorizer_entity_types.fit_transform([str(item['entity_types']) for item in train_features])
dev_vectors_entity_types = vectorizer_entity_types.transform([str(item['entity_types']) for item in dev_features])
test_vectors_entity_types = vectorizer_entity_types.transform([str(item['entity_types']) for item in test_features])

In [ ]:
vectorizer_entity_mention_type = TfidfVectorizer()
train_vectors_entity_mention_type = vectorizer_entity_mention_type.fit_transform([str(item['entity_mention_type']) for item in train_features])
dev_vectors_entity_mention_type = vectorizer_entity_mention_type.transform([str(item['entity_mention_type']) for item in dev_features])
test_vectors_entity_mention_type = vectorizer_entity_mention_type.transform([str(item['entity_mention_type']) for item in test_features])


##Combine features, prepare labels

In [ ]:
from scipy.sparse import hstack
X_train = hstack([train_vectors_words, train_vectors_pos, train_vectors_entity_types, train_vectors_entity_mention_type])
X_dev = hstack([dev_vectors_words, dev_vectors_pos, dev_vectors_entity_types, dev_vectors_entity_mention_type])
X_test = hstack([test_vectors_words, test_vectors_pos, test_vectors_entity_types, test_vectors_entity_mention_type])

In [ ]:
print("# of features:", X_train.shape[1])

# of features: 55243


In [ ]:
y_train = [item['label'] for item in train_features]
y_dev = [item['label'] for item in dev_features]
y_test = [item['label'] for item in test_features]

##Linear SVM classifier

In [ ]:

svm_classifier_linear = LinearSVC(random_state=42, C=0.1) #adjust C???
svm_classifier_linear.fit(X_train, y_train)

y_pred = svm_classifier_linear.predict(X_test)
print("Linear Kernel ===")
print(classification_report(y_test, y_pred))

Linear Kernel ===
                                     precision    recall  f1-score   support

                        no_relation       0.70      0.93      0.80      7770
                org:alternate_names       0.98      0.32      0.48       337
                 org:city_of_branch       0.84      0.25      0.38       129
              org:country_of_branch       0.65      0.18      0.28       166
                      org:dissolved       0.00      0.00      0.00         5
                        org:founded       1.00      0.09      0.16        34
                     org:founded_by       0.00      0.00      0.00        84
                      org:member_of       0.08      0.02      0.03        64
                        org:members       0.00      0.00      0.00        63
    org:number_of_employees/members       1.00      0.08      0.14        13
org:political/religious_affiliation       0.65      0.59      0.62        29
                   org:shareholders       0.00      0.00 

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Apply class weights to handle imbalance (fixed C=.1 parameter)

In [ ]:
# UNEVEN MARGINS
from sklearn.utils import class_weight
import numpy as np

#calculates class weights to balance nfluence of each class (based on num of samples)
class_weights = class_weight.compute_class_weight(
                                           class_weight='balanced',
                                           classes=np.unique(y_train),
                                           y=y_train)

#more fine-grained control, you can provide a dictionary mapping class labels to specific weights:
class_weights_dict = dict(zip(np.unique(y_train), class_weights))

#see weights for each class
print("Class weights: ", class_weights_dict)

#c is the regularization strength, multiply this times the class weights for each class
#higher class weight = less regularlization
svm_classifier_uneven = LinearSVC(random_state=42, C=0.1, class_weight=class_weights_dict)

svm_classifier_uneven.fit(X_train, y_train)

y_pred = svm_classifier_uneven.predict(X_test)

print("Uneven margins (linear) ===")
print(classification_report(y_test, y_pred))

Class weights:  {'no_relation': 0.03770865044761487, 'org:alternate_names': 1.108131159969674, 'org:city_of_branch': 2.349879421221865, 'org:country_of_branch': 1.6404320987654322, 'org:dissolved': 63.54891304347826, 'org:founded': 18.2703125, 'org:founded_by': 13.660046728971963, 'org:member_of': 4.004452054794521, 'org:members': 2.610044642857143, 'org:number_of_employees/members': 27.06712962962963, 'org:political/religious_affiliation': 7.692763157894737, 'org:shareholders': 15.716397849462366, 'org:stateorprovince_of_branch': 4.640079365079365, 'org:top_members/employees': 0.9909322033898305, 'org:website': 12.282563025210084, 'per:age': 3.471793349168646, 'per:cause_of_death': 12.821271929824562, 'per:charges': 16.80028735632184, 'per:children': 5.315, 'per:cities_of_residence': 7.774601063829787, 'per:city_of_birth': 16.609375, 'per:city_of_death': 12.180208333333333, 'per:countries_of_residence': 7.271766169154229, 'per:country_of_birth': 58.465, 'per:country_of_death': 243.604

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##Sample for non linear kernels

In [ ]:
from sklearn.model_selection import train_test_split

# 20% sample of the data for quick kernel experimentation
X_train_sample, _, y_train_sample, _ = train_test_split(
    X_train, y_train, train_size=0.1, random_state=42
)

# sparse matrix
print("Sampled x_train shape:", X_train_sample.shape)
# y_train_sample
print("Sampled y_train length:", len(y_train_sample))


Sampled x_train shape: (5846, 55243)
Sampled y_train length: 5846


In [ ]:
!nvidia-smi

Wed Feb 26 15:11:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       2MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!git clone https://github.com/rapidsai/rapidsai-csp-utils.git
!python rapidsai-csp-u

Cloning into 'rapidsai-csp-utils'...
remote: Enumerating objects: 582, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (66/66), done.
remote: Total 582 (delta 119), reused 82 (delta 82), pack-reused 434 (from 3)
Receiving objects: 100% (582/582), 190.86 KiB | 972.00 KiB/s, done.
Resolving deltas: 100% (293/293), done.
python3: can't open file '/content/rapidsai-csp-u': [Errno 2] No such file or directory


In [ ]:
!pip install cuml
import cuml
from cuml.svm import SVC

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for cuml
  Running setup.py clean for cuml
Failed to build cuml
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (cuml)


ModuleNotFoundError: No module named 'cuml'

##Testing cuML implementation

In [ ]:
# !pip install cudf-cu11 cuml-cu11 dask-cudf-cu11 --extra-index-url=https://pypi.ngc.nvidia.com
#TODO - get GPU version of SVM going

import cudf
import cupy as cp
from cuml.svm import SVC
from cuml.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split


X_train_sample_cupy = cp.sparse.csr_matrix(X_train_sample.todense())
X_test_cupy = cp.sparse.csr_matrix(X_test.todense())
y_train_sample_cudf = cudf.Series(y_train_sample)

y_test_cudf = cudf.Series(y_test)

svm_rbf_cuml = SVC(kernel='rbf', class_weight='balanced', random_state=42)
svm_rbf_cuml.fit(X_train_sample_cupy, y_train_sample_cudf)
y_pred_cuml = svm_rbf_cuml.predict(X_test_cupy).to_numpy()
=
print("RBF Kernel (cuML) ===")
print(classification_report(y_test, y_pred_cuml))

ModuleNotFoundError: No module named 'cuml'

## RBF kernel

In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report

svm_rbf = SVC(kernel='rbf', class_weight='balanced', random_state=42, verbose=True)

svm_rbf.fit(X_train_sample, y_train_sample)
y_pred = svm_rbf.predict(X_test)

print("RBF Kernel ===")
print(classification_report(y_test, y_pred))


[LibSVM]RBF Kernel ===
                                     precision    recall  f1-score   support

                        no_relation       0.76      0.66      0.71      7770
                org:alternate_names       0.67      0.73      0.70       337
                 org:city_of_branch       0.40      0.22      0.29       129
              org:country_of_branch       0.46      0.63      0.53       166
                      org:dissolved       0.00      0.00      0.00         5
                        org:founded       1.00      0.03      0.06        34
                     org:founded_by       0.00      0.00      0.00        84
                      org:member_of       0.00      0.00      0.00        64
                        org:members       0.05      0.06      0.06        63
    org:number_of_employees/members       0.00      0.00      0.00        13
org:political/religious_affiliation       0.00      0.00      0.00        29
                   org:shareholders       0.00      

/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


##Cosine Similarity Kernel

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.svm import SVC

#cosine similarity matrix for the training set
K_train = cosine_similarity(X_train_sample, X_train_sample)
K_test = cosine_similarity(X_test, X_train_sample)

#precomputed kernel in SVC
svm_cosine = SVC(kernel='precomputed', class_weight='balanced', random_state=42)
svm_cosine.fit(K_train, y_train)

y_pred = svm_cosine.predict(K_test)
from sklearn.metrics import classification_report
print("Cosine Similarity Kernel ===")
print(classification_report(y_test, y_pred))


ValueError: Found input variables with inconsistent numbers of samples: [5846, 58465]

## Chi-Squared Kernel

In [ ]:
from sklearn.metrics.pairwise import chi2_kernel

K_train = chi2_kernel(X_train, X_train, gamma=0.5)
K_test = chi2_kernel(X_test, X_train, gamma=0.5)

svm_chi2 = SVC(kernel='precomputed', class_weight='balanced', random_state=42)
svm_chi2.fit(K_train, y_train)

y_pred = svm_chi2.predict(K_test)
print("Chi-Square Kernel ===")
print(classification_report(y_test, y_pred))


##Laplacian Kernel

In [ ]:
from sklearn.metrics.pairwise import laplacian_kernel

K_train = laplacian_kernel(X_train, X_train, gamma=0.5)
K_test = laplacian_kernel(X_test, X_train, gamma=0.5)

svm_laplacian = SVC(kernel='precomputed', class_weight='balanced', random_state=42)
svm_laplacian.fit(K_train, y_train)

y_pred = svm_laplacian.predict(K_test)
print("Laplacian Kernel ===")
print(classification_report(y_test, y_pred))


##Can make custom kernel - can use own equation, combine kernels, etc

In [ ]:
import numpy as np

#custom polynomial kernel
def custom_poly_kernel(X, Y, degree=3, coef0=1):
    return (np.dot(X, Y.T) + coef0) ** degree

#compute custom kernel matrix
K_train = custom_poly_kernel(X_train, X_train, degree=4)
K_test = custom_poly_kernel(X_test, X_train, degree=4)

svm_custom_poly = SVC(kernel='precomputed', class_weight='balanced', random_state=42)
svm_custom_poly.fit(K_train, y_train)

y_pred = svm_custom_poly.predict(K_test)
print("Custom Polynomial Kernel (degree=4) ===")
print(classification_report(y_test, y_pred))


##TODO - custom subsequence kernel